# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis & Time Window

* **Unit of Analysis (Grain):** One row represents one unique content item per client (`client_hash_id` $\times$ `content_hash_id`), aggregated across the selected feature window.
* **Time Window:** A 30-day historical window covering **March 1, 2026 to March 31, 2026** (`date >= '2026-03-01' AND date <= '2026-03-31'`).

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Verify uniqueness at the stated grain (client_hash_id x content_hash_id for March 2026)
grain_check_query = """
WITH march_aggregated AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_sum_position) / SUM(gsc_impressions) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(*) - COUNT(DISTINCT content_hash_id) AS duplicate_check
FROM march_aggregated;
"""

# Ensure HF secret or setup is initialized prior if connecting remotely
grain_results = con.execute(grain_check_query).df()
print(grain_results)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_items  duplicate_check
0      331437                331437                0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: Feature / Label / Context / Excluded

### Label (Target Metric)
* `total_impressions`: Sum of `gsc_impressions` aggregated over the 30-day window. Represents how often the content appeared in search engine results.

### Features
* **Search Performance & Rank Signals:**
  * `total_clicks`: Sum of `gsc_clicks` aggregated over the 30-day window.
  * `avg_position`: Weighted average rank position derived from `SUM(gsc_sum_position) / SUM(gsc_impressions)`.
  * `active_days`: Count of distinct `report_date` instances where the content had recorded activity (`COUNT(DISTINCT report_date)`).
  * `historical_ctr`: Derived click-through rate calculated as `SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)`.
* **User Engagement & Analytics Signals (GA4):**
  * `total_pageviews`: Sum of `ga4_pageviews` over the 30-day window.
  * `total_sessions`: Sum of `ga4_sessions` over the 30-day window.
  * `engagement_rate`: Derived metric from GA4 computed as `SUM(ga4_engaged_sessions) / NULLIF(SUM(ga4_sessions), 0)`.
  * `avg_engagement_time`: Engagement duration per user calculated as `SUM(ga4_total_engagement_sec) / NULLIF(SUM(ga4_users), 0)`.
  * `organic_session_ratio`: Proportion of traffic from search computed as `SUM(sessions_organic) / NULLIF(SUM(ga4_sessions), 0)`.
  * `ai_referral_sessions`: Combined traffic from AI search engines (`SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other)`).

### Context
* `client_hash_id`: Anonymized unique identifier for each client.
* `content_hash_id`: Anonymized unique identifier for each content piece.
* `client_has_gsc`: Boolean indicator verifying whether Google Search Console tracking is enabled for the client.
* `client_has_ga4`: Boolean indicator verifying whether GA4 analytics tracking is enabled for the client.

### Excluded
* `report_date`: Excluded because raw daily timestamps violate the aggregated 30-day unit of analysis (`client_hash_id` $\times$ `content_hash_id`).
* `gsc_sum_position`: Excluded as a raw feature because it is an unnormalized running sum; used only internally to derive the normalized `avg_position`.
* Individual daily referral columns (`ai_chatgpt`, `ai_perplexity`, etc.): Excluded in raw form to reduce noise; aggregated into a combined AI traffic feature (`ai_referral_sessions`).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 1. Grain Query

In [5]:
# 1. Grain Query
grain_query = """
WITH aggregated_data AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '_' || content_hash_id) AS unique_pairs,
    COUNT(*) - COUNT(DISTINCT client_hash_id || '_' || content_hash_id) AS duplicate_count
FROM aggregated_data;
"""
print(con.execute(grain_query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pairs  duplicate_count
0      331437        331437                0


Verified that duplicate_count is 0, confirming my grain holds without duplicates

### Time Window/Date Bounds Check

In [6]:
# 2. Window Check
window_query = """
SELECT 
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS active_days
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31';
"""
print(con.execute(window_query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  active_days
0 2026-03-01 2026-03-31           31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.